# One-parameter SMC-ABC for cooperativity energy with hopping turned off

This notebook does a **one-parameter inference** for `cooperativity_energy` using the spatial PRC1 Gillespie model, with:

- all other kinetic parameters fixed,
- `enable_cooperativity=True`,
- `enable_hopping=False`.

Before running SMC-ABC, it performs **one timing check** for a single simulation so you can gauge cost.

## 1. Load local modules

This loader assumes the model files are in the same folder as this notebook, or in a nearby working directory.
It loads:

- `gillespie.py`
- `prc1.py`
- `prc1_state.py`
- `run_gillespie.py`
- `smc-abc.py` or `smc_abc.py`

In [ ]:
from pathlib import Path
import importlib.util
import sys

def load_module(module_name, filename_candidates):
    search_roots = [
        Path.cwd(),
        Path.cwd() / "python_version",
        Path.cwd().parent,
    ]
    for root in search_roots:
        for name in filename_candidates:
            path = root / name
            if path.exists():
                spec = importlib.util.spec_from_file_location(module_name, path)
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)
                print(f"Loaded {module_name} from: {path}")
                return module
    raise FileNotFoundError(f"Could not find any of: {filename_candidates}")

gillespie = load_module("gillespie", ["gillespie.py"])
prc1 = load_module("prc1", ["prc1.py"])
prc1_state = load_module("prc1_state", ["prc1_state.py"])
run_gillespie = load_module("run_gillespie", ["run_gillespie.py"])
smc_abc = load_module("smc_abc", ["smc_abc.py", "smc-abc.py"])

## 2. Fixed parameters and observation grid

In [ ]:
import numpy as np

# fixed model parameters
INIT_BIND = 0.002
K_DET_1 = 0.15
K_ATT_2 = 0.8
K_DET_2 = 0.1
MAX_STEPS = 200_000

# observation grid
times_obs = np.arange(0.0, 68.0 + 4.0, 4.0)

# thermal scale
k_B_T = 4.1

# true cooperativity energy used for synthetic data
EPS_TRUE = 2.3 * k_B_T

print("times_obs =", times_obs)
print("EPS_TRUE =", EPS_TRUE)

## 3. Define a one-parameter simulator

Here `theta[0]` is the cooperativity energy.  
Everything else is fixed, and hopping is disabled.

In [ ]:
def simulate_one_coop_only(theta, times_obs, seed):
    """
    theta[0] = cooperativity_energy
    All other parameters are fixed.
    Hopping is disabled.
    """
    import numpy as np
    np.random.seed(int(seed))

    return run_gillespie.run_gillespie_prc1_on_grid(
        initial_binding_rate_per_site=INIT_BIND,
        singly_bound_detachment_rate=K_DET_1,
        base_double_attachment_rate=K_ATT_2,
        base_double_detachment_rate=K_DET_2,
        times_obs=times_obs,
        cooperativity_energy=float(theta[0]),
        enable_cooperativity=True,
        enable_hopping=False,
        max_steps=MAX_STEPS,
    )

## 4. Time one simulation before doing inference

This is the one-time timing check you asked for.

In [ ]:
import time

theta_test = np.array([EPS_TRUE], dtype=float)

t0 = time.perf_counter()
test_path = simulate_one_coop_only(theta_test, times_obs, seed=123)
t1 = time.perf_counter()

one_sim_seconds = t1 - t0

print("One simulation path:")
print(test_path)
print("Shape:", test_path.shape)
print(f"Time for one simulation: {one_sim_seconds:.6f} seconds")

## 5. Patch the SMC-ABC module to use the one-parameter simulator

In [ ]:
smc_abc.simulate_one = simulate_one_coop_only
print("Patched smc_abc.simulate_one successfully.")

## 6. Generate synthetic observed data

This creates fake observed data from the true cooperativity energy, still with hopping turned off.

In [ ]:
n_obs_reps = 3

y_obs = np.array([
    run_gillespie.run_gillespie_prc1_on_grid(
        initial_binding_rate_per_site=INIT_BIND,
        singly_bound_detachment_rate=K_DET_1,
        base_double_attachment_rate=K_ATT_2,
        base_double_detachment_rate=K_DET_2,
        times_obs=times_obs,
        cooperativity_energy=EPS_TRUE,
        enable_cooperativity=True,
        enable_hopping=False,
        max_steps=MAX_STEPS,
    )
    for _ in range(n_obs_reps)
], dtype=float)

print("y_obs shape:", y_obs.shape)
print(y_obs)

## 7. Set a one-parameter prior in log-space

In [ ]:
phi_mu = np.array([np.log(EPS_TRUE)], dtype=float)
phi_sd = np.array([0.5], dtype=float)

print("phi_mu =", phi_mu)
print("phi_sd =", phi_sd)

## 8. Run SMC-ABC

The settings below are intentionally modest for a first pass.
You can increase `P`, `G`, `pool`, and `n_reps` later.

In [ ]:
res = smc_abc.smc_abc_prc1(
    times_obs=times_obs,
    y_obs=y_obs,
    phi_mu=phi_mu,
    phi_sd=phi_sd,
    P=20,
    G=3,
    pool=60,
    n_reps=2,
    eps_quantile=50.0,
    cov_scale=2.0,
    seed=1,
    n_jobs=-1,
    batch_factor=4,
)

print("Finished SMC-ABC.")
print("eps history:", res.eps_history)

## 9. Convert posterior particles back to energy scale

In [ ]:
posterior_eps = np.exp(res.particles_phi[:, 0])
posterior_w = res.weights

print("posterior particles:")
print(posterior_eps)
print("posterior weights:")
print(posterior_w)

## 10. Posterior summaries

In [ ]:
post_mean = np.sum(posterior_w * posterior_eps)

order = np.argsort(posterior_eps)
eps_sorted = posterior_eps[order]
w_sorted = posterior_w[order]
cdf = np.cumsum(w_sorted)
post_median = eps_sorted[np.searchsorted(cdf, 0.5)]

print("True cooperativity energy:", EPS_TRUE)
print("Posterior mean:", post_mean)
print("Posterior median:", post_median)

## 11. Posterior plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(posterior_eps, bins=10, weights=posterior_w)
plt.axvline(EPS_TRUE, linestyle="--", label="true value")
plt.xlabel("cooperativity energy")
plt.ylabel("posterior weight")
plt.title("One-parameter posterior with hopping off")
plt.legend()
plt.show()

## Notes

- This notebook assumes the loaded `run_gillespie.py` supports:
  - `initial_binding_rate`
  - `enable_cooperativity`
  - `enable_hopping`
- It also assumes `smc_abc.smc_abc_prc1(...)` uses `smc_abc.simulate_one(...)` internally, so monkey-patching works.
- Run the timing cell first. If one simulation is already too slow, lower `MAX_STEPS` or reduce the grid length before SMC-ABC.